# JAX vs NumPy Performance Comparison

This notebook compares the performance of lexicase selection implementations using JAX vs NumPy arrays across different problem sizes.

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
from lexicase import (
    lexicase_selection, 
    epsilon_lexicase_selection, 
    downsample_lexicase_selection,
    informed_downsample_lexicase_selection
)

# Check if JAX is available
try:
    import jax
    import jax.numpy as jnp
    JAX_AVAILABLE = True
    print("JAX is available")
    print(f"JAX version: {jax.__version__}")
    print(f"JAX devices: {jax.devices()}")
except ImportError:
    JAX_AVAILABLE = False
    print("JAX is not available - only NumPy benchmarks will run")

## Benchmark Functions

In [ ]:
def generate_fitness_data(n_individuals, n_cases, array_type='numpy', seed=42):
    """
    Generate random fitness data for benchmarking.
    
    Args:
        n_individuals: Number of individuals
        n_cases: Number of test cases
        array_type: 'numpy' or 'jax'
        seed: Random seed
        
    Returns:
        Fitness matrix as numpy or JAX array
    """
    np.random.seed(seed)
    fitness_matrix = np.random.uniform(0, 100, (n_individuals, n_cases))
    
    if array_type == 'jax' and JAX_AVAILABLE:
        fitness_matrix = jnp.array(fitness_matrix)
    
    return fitness_matrix


def benchmark_function(func, *args, n_runs=10, warmup_runs=2):
    """
    Benchmark a function with multiple runs.
    
    Args:
        func: Function to benchmark
        *args: Arguments to pass to function
        n_runs: Number of timing runs
        warmup_runs: Number of warmup runs (not timed)
        
    Returns:
        tuple: (mean_time, std_time, times_list)
    """
    # Warmup runs
    for _ in range(warmup_runs):
        result = func(*args)
        # For JAX, force computation
        if hasattr(result, 'block_until_ready'):
            result.block_until_ready()
    
    # Timing runs
    times = []
    for _ in range(n_runs):
        start_time = time.perf_counter()
        result = func(*args)
        # For JAX, force computation
        if hasattr(result, 'block_until_ready'):
            result.block_until_ready()
        end_time = time.perf_counter()
        times.append(end_time - start_time)
    
    return np.mean(times), np.std(times), times

## Single Size Comparison

Let's compare NumPy and JAX performance at a single problem size first.

In [ ]:
# Test with a single problem size first
n_individuals = 500
n_cases = 30
num_selected = 50

print(f"Benchmarking with {n_individuals} individuals, {n_cases} cases, selecting {num_selected}")
print("=" * 70)

# Generate test data
numpy_data = generate_fitness_data(n_individuals, n_cases, 'numpy')
if JAX_AVAILABLE:
    jax_data = generate_fitness_data(n_individuals, n_cases, 'jax')

# Benchmark standard lexicase selection
print("\nStandard Lexicase Selection:")
numpy_time, numpy_std, _ = benchmark_function(
    lexicase_selection, numpy_data, num_selected, 42
)
print(f"NumPy: {numpy_time*1000:.2f} +/- {numpy_std*1000:.2f} ms")

if JAX_AVAILABLE:
    jax_time, jax_std, _ = benchmark_function(
        lexicase_selection, jax_data, num_selected, 42
    )
    print(f"JAX:   {jax_time*1000:.2f} +/- {jax_std*1000:.2f} ms")
    speedup = numpy_time / jax_time
    faster = "JAX" if speedup > 1 else "NumPy"
    print(f"Speedup: {speedup:.2f}x ({faster} faster)")

# Benchmark epsilon lexicase selection
print("\nEpsilon Lexicase Selection (MAD):")
numpy_eps_time, numpy_eps_std, _ = benchmark_function(
    epsilon_lexicase_selection, numpy_data, num_selected, None, 42
)
print(f"NumPy: {numpy_eps_time*1000:.2f} +/- {numpy_eps_std*1000:.2f} ms")

if JAX_AVAILABLE:
    jax_eps_time, jax_eps_std, _ = benchmark_function(
        epsilon_lexicase_selection, jax_data, num_selected, None, 42
    )
    print(f"JAX:   {jax_eps_time*1000:.2f} +/- {jax_eps_std*1000:.2f} ms")
    eps_speedup = numpy_eps_time / jax_eps_time
    faster = "JAX" if eps_speedup > 1 else "NumPy"
    print(f"Speedup: {eps_speedup:.2f}x ({faster} faster)")

# Benchmark downsampled lexicase selection
print("\nDownsampled Lexicase Selection:")
downsample_size = min(10, n_cases)
numpy_down_time, numpy_down_std, _ = benchmark_function(
    downsample_lexicase_selection, numpy_data, num_selected, downsample_size, 42
)
print(f"NumPy: {numpy_down_time*1000:.2f} +/- {numpy_down_std*1000:.2f} ms")

if JAX_AVAILABLE:
    jax_down_time, jax_down_std, _ = benchmark_function(
        downsample_lexicase_selection, jax_data, num_selected, downsample_size, 42
    )
    print(f"JAX:   {jax_down_time*1000:.2f} +/- {jax_down_std*1000:.2f} ms")
    down_speedup = numpy_down_time / jax_down_time
    faster = "JAX" if down_speedup > 1 else "NumPy"
    print(f"Speedup: {down_speedup:.2f}x ({faster} faster)")

## Scaling Performance Analysis

Let's see how performance scales with problem size.

In [ ]:
# Test scaling across different problem sizes
problem_sizes = [
    (100, 10, 20),
    (250, 20, 50),
    (500, 30, 100),
    (1000, 50, 200),
    (2000, 100, 400),
]

numpy_times = []
jax_times = []
problem_labels = []

print("Scaling Analysis - Standard Lexicase Selection")
print("Problem Size (individuals x cases -> selected) | NumPy Time | JAX Time | Speedup")
print("=" * 80)

for n_ind, n_cases, n_sel in problem_sizes:
    problem_labels.append(f"{n_ind}x{n_cases}")
    
    # Generate data
    numpy_data = generate_fitness_data(n_ind, n_cases, 'numpy')
    
    # Benchmark NumPy
    numpy_time, _, _ = benchmark_function(
        lexicase_selection, numpy_data, n_sel, 42, n_runs=5
    )
    numpy_times.append(numpy_time)
    
    if JAX_AVAILABLE:
        jax_data = generate_fitness_data(n_ind, n_cases, 'jax')
        jax_time, _, _ = benchmark_function(
            lexicase_selection, jax_data, n_sel, 42, n_runs=5
        )
        jax_times.append(jax_time)
        speedup = numpy_time / jax_time
        
        print(f"{n_ind:4d} x {n_cases:3d} -> {n_sel:3d}                     | "
              f"{numpy_time*1000:8.1f}ms | {jax_time*1000:7.1f}ms | {speedup:6.2f}x")
    else:
        jax_times.append(numpy_time)  # Fallback for plotting
        print(f"{n_ind:4d} x {n_cases:3d} -> {n_sel:3d}                     | "
              f"{numpy_time*1000:8.1f}ms | N/A      | N/A")

In [ ]:
# Plot scaling results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Absolute times
x_pos = np.arange(len(problem_labels))
width = 0.35

bars1 = ax1.bar(x_pos - width/2, [t*1000 for t in numpy_times], width, 
                label='NumPy', alpha=0.8, color='blue')
if JAX_AVAILABLE:
    bars2 = ax1.bar(x_pos + width/2, [t*1000 for t in jax_times], width, 
                    label='JAX', alpha=0.8, color='orange')

ax1.set_xlabel('Problem Size (individuals x cases)')
ax1.set_ylabel('Time (ms)')
ax1.set_title('Absolute Performance Comparison')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(problem_labels, rotation=45)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Speedup ratio
if JAX_AVAILABLE:
    speedups = [numpy_times[i] / jax_times[i] for i in range(len(numpy_times))]
    colors = ['green' if s > 1 else 'red' for s in speedups]
    bars3 = ax2.bar(x_pos, speedups, alpha=0.8, color=colors)
    ax2.axhline(y=1, color='black', linestyle='--', alpha=0.7, label='No speedup')
    ax2.set_xlabel('Problem Size (individuals x cases)')
    ax2.set_ylabel('Speedup (NumPy time / JAX time)')
    ax2.set_title('JAX Speedup Over NumPy\n(Green = JAX faster, Red = NumPy faster)')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(problem_labels, rotation=45)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Add speedup labels
    for bar, speedup in zip(bars3, speedups):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.05,
                 f'{speedup:.2f}x', ha='center', va='bottom', fontsize=9, fontweight='bold')
else:
    ax2.text(0.5, 0.5, 'JAX not available\nfor comparison', 
             ha='center', va='center', transform=ax2.transAxes, fontsize=14)
    ax2.set_title('Speedup Analysis (JAX not available)')

plt.tight_layout()
plt.show()

## Detailed Comparison by Selection Type

Let's compare all selection methods at a medium scale.

In [ ]:
# Compare all selection types at medium scale
medium_n_individuals = 500
medium_n_cases = 30
medium_num_selected = 100
downsample_size = 10

print(f"Detailed Comparison ({medium_n_individuals} individuals, {medium_n_cases} cases)")
print("=" * 70)

# Generate test data
numpy_data = generate_fitness_data(medium_n_individuals, medium_n_cases, 'numpy')
if JAX_AVAILABLE:
    jax_data = generate_fitness_data(medium_n_individuals, medium_n_cases, 'jax')

selection_methods = [
    ('Standard Lexicase', 
     lambda data: lexicase_selection(data, medium_num_selected, 42)),
    ('Epsilon Lexicase (MAD)', 
     lambda data: epsilon_lexicase_selection(data, medium_num_selected, None, 42)),
    ('Epsilon Lexicase (e=1.0)', 
     lambda data: epsilon_lexicase_selection(data, medium_num_selected, 1.0, 42)),
    ('Downsampled Lexicase', 
     lambda data: downsample_lexicase_selection(data, medium_num_selected, downsample_size, 42)),
    ('Informed Downsample', 
     lambda data: informed_downsample_lexicase_selection(data, medium_num_selected, downsample_size, 42)),
]

results = []

for method_name, method_func in selection_methods:
    print(f"\n{method_name}:")
    
    # NumPy timing
    numpy_time, numpy_std, _ = benchmark_function(method_func, numpy_data, n_runs=5)
    print(f"  NumPy: {numpy_time*1000:.1f} +/- {numpy_std*1000:.1f} ms")
    
    if JAX_AVAILABLE:
        # JAX timing
        jax_time, jax_std, _ = benchmark_function(method_func, jax_data, n_runs=5)
        print(f"  JAX:   {jax_time*1000:.1f} +/- {jax_std*1000:.1f} ms")
        
        speedup = numpy_time / jax_time
        faster = "JAX" if speedup > 1 else "NumPy"
        print(f"  Speedup: {speedup:.2f}x ({faster} faster)")
        
        results.append((method_name, numpy_time*1000, jax_time*1000, speedup))
    else:
        print(f"  JAX: Not available")
        results.append((method_name, numpy_time*1000, numpy_time*1000, 1.0))

In [ ]:
# Plot comparison by method
if results:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    methods = [r[0] for r in results]
    numpy_times = [r[1] for r in results]
    jax_times = [r[2] for r in results]
    speedups = [r[3] for r in results]
    
    x_pos = np.arange(len(methods))
    width = 0.35
    
    # Timing comparison
    ax1.bar(x_pos - width/2, numpy_times, width, label='NumPy', alpha=0.8)
    if JAX_AVAILABLE:
        ax1.bar(x_pos + width/2, jax_times, width, label='JAX', alpha=0.8)
    
    ax1.set_xlabel('Selection Method')
    ax1.set_ylabel('Time (ms)')
    ax1.set_title('Performance by Selection Method')
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(methods, rotation=30, ha='right')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Speedup comparison
    colors = ['green' if s > 1 else 'red' for s in speedups]
    bars = ax2.bar(x_pos, speedups, alpha=0.8, color=colors)
    ax2.axhline(y=1, color='black', linestyle='--', alpha=0.7, label='No speedup')
    ax2.set_xlabel('Selection Method')
    ax2.set_ylabel('Speedup (NumPy / JAX)')
    ax2.set_title('JAX Speedup by Method')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(methods, rotation=30, ha='right')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Add speedup labels
    for bar, speedup in zip(bars, speedups):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                 f'{speedup:.2f}x', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()

## Summary and Recommendations

In [ ]:
print("PERFORMANCE SUMMARY")
print("=" * 50)

if JAX_AVAILABLE:
    print("\nJAX Implementation Available")
    print("\nKey Findings:")
    print("  - NumPy is often faster for small to medium problems on CPU")
    print("  - JAX has JIT compilation overhead that impacts small problems")
    print("  - JAX excels with GPU acceleration (not demonstrated on CPU)")
    print("  - Automatic dispatch means you get the right backend automatically")
    
    print("\nRecommendations:")
    print("  - For CPU-only work: NumPy is typically faster")
    print("  - For GPU acceleration: Use JAX arrays")
    print("  - For repeated selection: JAX JIT benefits may help")
    print("  - Simply pass the right array type and dispatch handles the rest")
    
else:
    print("\nJAX Not Available")
    print("\nTo enable JAX:")
    print("  pip install jax jaxlib")
    print("  pip install jax[cuda]  # For NVIDIA GPU")
    print("\nThe NumPy implementation provides excellent performance for most use cases.")

print("\nAutomatic Dispatch Usage:")
print("```python")
print("import numpy as np")
print("from lexicase import lexicase_selection")
print("")
print("# NumPy arrays -> NumPy implementation")
print("numpy_fitness = np.random.random((1000, 50))")
print("numpy_result = lexicase_selection(numpy_fitness, 100, seed=42)")
print("")
print("# JAX arrays -> JAX implementation (if JAX installed)")
print("import jax.numpy as jnp")
print("jax_fitness = jnp.array(numpy_fitness)")
print("jax_result = lexicase_selection(jax_fitness, 100, seed=42)")
print("```")